<a href="https://colab.research.google.com/github/raffaellaalpaca-ai/ds2002-fa26/blob/main/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [13]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


In [14]:
#imported myself
import pandas as pd
# Load the real Walmart product catalog
cat = pd.read_csv("product_catalog.csv")

# Save the original number of rows
original_count = len(cat)

# Display the first 5 rows
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


In [15]:
#verifying rows
print("Total rows:", len(cat))
print("Total columns:", len(cat.columns))

Total rows: 45
Total columns: 7


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [16]:
# TODO (hint: .str.lower().str.contains('pop'))

#find products that contain "pop" in their name
pop_tarts = cat[cat["product_name"].str.lower().str.contains("pop", na=False)]
#count how many products we found
print("Number of Pop-Tart rows:", len(pop_tarts))
#show the matching products
print(pop_tarts)

Number of Pop-Tart rows: 4
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [18]:
#find strawberry Pop-Tarts
strawberry = pop_tarts[pop_tarts["product_name"].str.lower().str.contains("strawberry", na=False)]
# Show their SKUs and names
print(strawberry[["sku", "product_name"]])
#poptarts that are not strawberry
other_flavors = pop_tarts[
    ~pop_tarts["product_name"].str.lower().str.contains("strawberry", na=False)
]
print(other_flavors[["sku", "product_name"]])


             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** Pop-Tarts with different flavors should remain separate products because they are not identical to strawberry Pop-Tarts.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [21]:
# Create a dictionary to group the same products
sku_map = {
    "PT-12": "PT-12",
    "SKU#459812": "PT-12",
    "POPTART-STRAW": "PT-12",
    "PT_STRAWBERRY": "PT-12"  # Added this one!
}

# New column with the canonical SKU
cat["canonical_sku"] = cat["sku"].replace(sku_map)

# Show results
print(cat[["sku", "product_name", "canonical_sku"]])

              sku            product_name  canonical_sku
0           PT-12    pop-tarts strawberry          PT-12
1      SKU#459812     Strawberry Pop-Tart          PT-12
2   POPTART-STRAW    Pop-Tarts Strawberry          PT-12
3   PT_STRAWBERRY    pop-tarts strawberry          PT-12
4          WTR-24      BOTTLED WATER 24PK         WTR-24
5       WATER24PK   Bottled Water 24-Pack      WATER24PK
6        WTR_24PK         Water - 24 Pack       WTR_24PK
7        BATT-100        BATTERIES AA 4PK       BATT-100
8         BAT-AA4     AA Batteries 4-Pack        BAT-AA4
9     BATT_AA_4PK     AA Batteries 4-Pack    BATT_AA_4PK
10       FLASH-01          FLASHLIGHT LED       FLASH-01
11      LED-FLASH          FLASHLIGHT LED      LED-FLASH
12        BEER-24    Light Beer (24-pack)        BEER-24
13       LTBEER24    Light Beer (24-pack)       LTBEER24
14        BG-MONO     Monopoly Board Game        BG-MONO
15    MONOPOLY-01     Monopoly Board Game    MONOPOLY-01
16      GAME-MONO   Board Game 

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [24]:
# Combine categories that mean the same thing
cat["category"] = cat["category"].replace({
    "Breakfast Foods": "Breakfast",
    "Emergency Supplies": "Emergency",
    "Groceries": "Grocery",
    "Baby Products": "Baby",
    "Alcoholic Beverages": "Alcohol"
})

# Show categories after combining
print("After combining:")
print(cat["category"].value_counts())

After combining:
category
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby              4
Household         4
Alcohol           2
Snacks            2
Health            2
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [8]:
#function to clean product names
def clean_name(name):
    name = str(name).lower()
    #removing punctuation
    name = name.replace("-", " ")
    name = name.replace(",", " ")
    name = name.replace(".", " ")
    #sort words alphabetically
    words = name.split()
    words.sort()

    return " ".join(words)
#create a new column with cleaned names
cat["clean_name"] = cat["product_name"].apply(clean_name)

#group products by cleaned name
groups = cat.groupby("clean_name")

#show groups with more than one unique SKU
for name, group in groups:
    if group["sku"].nunique() > 1:
        print("\nPossible duplicates:", name)
        print(group[["sku", "product_name"]])


Possible duplicates: pop strawberry tarts
             sku          product_name
0          PT-12  pop-tarts strawberry
2  POPTART-STRAW  Pop-Tarts Strawberry


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [9]:
# Check that no canonical SKUs are missing
assert cat["canonical_sku"].isnull().sum() == 0
# Check that the number of rows stayed the same
assert len(cat) == len(pd.read_csv(PATH)) if __import__("os").path.exists(PATH) else len(cat) == 3
print("All checks passed!")

All checks passed!


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

_your answer here_